# Q1 — mdx vs WT: the disease signature in quadriceps (QUA)

**Question.** What distinguishes dystrophic (mdx) muscle from wild-type (WT) muscle at single-fiber resolution?

**Data.** Study 22-082, quadriceps (QUA), 19 samples (5 WT, 5 mdx, 4 AAV9, 5 LICA1), 111,403 fibers matched across serial sections. Each fiber carries 807 features: 15 morphological + 36 intensity statistics x 22 staining channels (7 slides).

**Clustering (precomputed, not re-run).** Fibers were clustered *unbiasedly* — on morphology (15 features) + HE brightfield (108 RGB features) only, i.e. without any fluorescence staining: z-score -> PCA(30) -> UMAP(nn=15, min_dist=0) -> GMM(k=6) on the PCA scores. This notebook loads the precomputed cache (built by `scripts/build_analysis_bundle.py`) and re-fits the deterministic GMM (random_state=42) — cluster sizes are identical to the original run.

**Caveat.** Slide 8 (LAMP2/LGALS3/SQSTM1) mapping failed for 3/4 AAV9 samples and 1/5 mdx samples; AAV9 is therefore excluded from all slide-8 analyses (see Q2 notebook for the per-sample detail).


In [1]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr, mannwhitneyu

REPO = Path("/DATA/F2FMatcher_DDC")
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))
from cluster_fibers import MARKERS, feature_observed_mask

BUNDLE = REPO / "results/QUA/analysis_bundle.npz"
OUT = REPO / "visualizations/Q1"
OUT.mkdir(parents=True, exist_ok=True)

z = np.load(BUNDLE, allow_pickle=True)
Xpre = z["Xpre"].astype(np.float64)          # 807 raw features, pre-imputation (NaN = not mapped)
X_umap = z["X_umap"].astype(np.float64)
groups = z["groups"]; samples = z["samples"]
clusters = z["cluster_ids"]                   # GMM k=6 labels 0..5
cols = [str(c) for c in z["cols"]]
n = len(X_umap)
idx = {c: i for i, c in enumerate(cols)}

GROUPS = ["WT", "mdx", "AAV9", "LICA1"]
GROUP_COLORS = {"WT": "#0000C0", "mdx": "#FF6000", "AAV9": "#C0C000", "LICA1": "#008000"}
CLUSTER_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]
CLUSTERS = [f"C{i+1}" for i in range(6)]

def mval(name):
    """Raw (pre-imputation) marker values + per-fiber observed mask (mapped & finite)."""
    spec = MARKERS[name]
    if isinstance(spec, str):
        v = Xpre[:, idx[spec]]
    else:
        v = np.mean([Xpre[:, idx[c]] for c in spec], axis=0)
    o = feature_observed_mask(name, Xpre, cols) & np.isfinite(v)
    return v, o

def umap_scatter(ax, m, color=None, s=1.0, alpha=0.3):
    ax.scatter(X_umap[m, 0], X_umap[m, 1], s=s, alpha=alpha, c=color,
               edgecolors="none", rasterized=True)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

print(f"fibers: {n:,}")
print(pd.Series(groups).value_counts().reindex(GROUPS))


fibers: 111,403
WT       34861
mdx      23057
AAV9     25662
LICA1    27823
Name: count, dtype: int64


## 1. Global landscape

The unbiased UMAP (morphology + HE only) already separates the groups: mdx fibers spread into regions with little WT support, while AAV9/LICA1 fibers shift back toward the WT core. The GMM (k=6, chosen by BIC + seed stability) defines 6 fiber states.


In [2]:

# Fig 1 — UMAP colored by group
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for g in GROUPS:
    umap_scatter(ax, groups == g, color=GROUP_COLORS[g])
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=GROUP_COLORS[g],
                  markersize=8, label=f"{g} (n={int((groups == g).sum()):,})") for g in GROUPS]
ax.legend(handles=handles, fontsize=10, loc="best")
ax.set_title("QUA fibers — UMAP (morphology + HE only), colored by group", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "fig1_umap_by_group.png", dpi=200)
plt.show()


In [3]:

# Fig 2 — UMAP colored by GMM cluster
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for c in range(6):
    umap_scatter(ax, clusters == c, color=CLUSTER_COLORS[c])
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=CLUSTER_COLORS[c],
                  markersize=8, label=f"{CLUSTERS[c]} (n={int((clusters == c).sum()):,})")
           for c in range(6)]
ax.legend(handles=handles, fontsize=10, loc="best")
ax.set_title("QUA fibers — UMAP, colored by GMM cluster (k=6)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "fig2_umap_by_cluster.png", dpi=200)
plt.show()


In [4]:

# Fig 3 — per-group UMAP panels colored by cluster
fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)
for ax, g in zip(axes.ravel(), GROUPS):
    m = groups == g
    for c in range(6):
        umap_scatter(ax, m & (clusters == c), color=CLUSTER_COLORS[c])
    ax.text(0.05, 0.95, f"{g} (n={int(m.sum()):,})", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
fig.tight_layout()
fig.savefig(OUT / "fig3_umap_groups_clusters.png", dpi=200)
plt.show()


## 2. Cluster identities

Clusters were defined without staining information; the marker profiles below are therefore *discovery*, not circular. Signature = mean global z of each marker inside the cluster.

| Cluster | n | composition WT/mdx/AAV9/LICA1 | signature |
|---|---|---|---|
| C1 | 13,818 | 26/26/22/26 | large, metabolically quiet (NADH -0.7, COX -0.8), Myh4-enriched |
| C2 | 8,701 | 14/40/25/21 | small, fibrotic + inflammatory (IgG +0.9, LAMP2 +1.0, Col4 +0.3) |
| C3 | 22,715 | 6/38/31/26 | small, metabolically active (NADH +1.0, COX +0.9, Myh7 +1.0), immune + autophagy |
| C4 | 20,765 | 62/4/15/19 | small, healthy, fast (Myh7 +1.0), intact BM (WGA +0.7, Laminin +0.4) |
| C5 | 16,563 | 1/29/34/36 | large, Myh4 +0.7, BM defect (Laminin -0.3), quiet (NADH -0.5) |
| C6 | 28,841 | 55/6/17/23 | large, quiet (NADH -0.9, COX -0.8), WT-dominant |


In [5]:

# Fig 4 — cluster signature: mean global z per marker per cluster (observed fibers only)
MARK = list(MARKERS)
zmat = np.full((6, len(MARK)), np.nan)
for j, name in enumerate(MARK):
    v, o = mval(name)
    mu, sd = np.nanmean(v[o]), np.nanstd(v[o])
    zv = np.full(n, np.nan)
    zv[o] = (v[o] - mu) / sd
    zmat[:, j] = [np.nanmean(zv[clusters == c]) for c in range(6)]

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(zmat, aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)
ax.set_xticks(range(len(MARK)), MARK, rotation=60, fontsize=10)
ax.set_yticks(range(6), CLUSTERS, fontsize=11)
for i in range(6):
    for j in range(len(MARK)):
        if np.isfinite(zmat[i, j]):
            ax.text(j, i, f"{zmat[i, j]:+.1f}", ha="center", va="center",
                    fontsize=7, color="black" if abs(zmat[i, j]) < 0.9 else "white")
ax.set_title("Cluster signature (z vs all other fibers)", fontweight="bold")
fig.colorbar(im, label="global z")
fig.tight_layout()
fig.savefig(OUT / "fig4_cluster_signature.png", dpi=200)
plt.show()


## 3. Does mdx change the fiber-state composition?

Yes, strongly: mdx loses the two healthy small-fiber states (C4: 37% -> 4% of fibers, C6: 45% -> 7%) and gains the pathological states (C2: 4% -> 15%, C3: 4% -> 38%, C5: 0.5% -> 21%).


In [6]:

# Fig 5 — cluster composition: share of each group's fibers falling in each cluster (in-group %)
rows = []
for c in range(6):
    mc = clusters == c
    for g in ["WT", "mdx"]:
        rows.append(dict(cluster=CLUSTERS[c], group=g,
                         pct=100 * (mc & (groups == g)).sum() / (groups == g).sum()))
comp = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
w = 0.38
for k, g in enumerate(["WT", "mdx"]):
    d = comp[comp.group == g]
    ax.bar(np.arange(6) + (k - 0.5) * w, d.pct, width=w, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
    for i, v in enumerate(d.pct):
        ax.text(i + (k - 0.5) * w, v + 0.4, f"{v:.0f}", ha="center", fontsize=9)
ax.set_xticks(range(6), CLUSTERS, fontsize=11)
ax.set_ylabel("% of group's fibers in cluster")
ax.set_title("mdx vs WT — cluster composition (in-group fraction)", fontweight="bold")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "fig5_composition_wt_mdx.png", dpi=200)
plt.show()
comp.pivot(index="cluster", columns="group", values="pct").round(1)


group,WT,mdx
cluster,,
C1,10.2,15.4
C2,3.6,15.0
C3,3.8,37.6
C4,36.7,4.0
C5,0.5,20.7
C6,45.1,7.3


## 4. The mdx disease signature, per cluster

Cliff's delta of mdx vs WT per cluster x marker (red = higher in mdx). Robust across all clusters: **dystrophin down, IgG/CD11b up, LAMP2 up**. Cluster-specific: C3/C4 additionally lose Myh7 (fast) and WGA (membrane); C6 shows hypertrophy (area up) and nuclear changes (DAPI down in cyto2).


In [7]:

# Fig 6 — mdx vs WT per cluster x marker: Cliff's delta (effect size, Mann-Whitney)
MARK = list(MARKERS)
mat = np.full((6, len(MARK)), np.nan)
for i in range(6):
    mc = clusters == i
    for j, name in enumerate(MARK):
        v, o = mval(name)
        x = v[o & mc & (groups == "WT")]
        y = v[o & mc & (groups == "mdx")]
        if len(x) < 10 or len(y) < 10:
            continue
        less = (x[:, None] < y[None, :]).sum()
        great = (x[:, None] > y[None, :]).sum()
        mat[i, j] = (great - less) / (len(x) * len(y))

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(mat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(MARK)), MARK, rotation=60, fontsize=10)
ax.set_yticks(range(6), CLUSTERS, fontsize=11)
for i in range(6):
    for j in range(len(MARK)):
        if np.isfinite(mat[i, j]):
            ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=7,
                    color="black" if abs(mat[i, j]) < 0.6 else "white")
ax.set_title("mdx vs WT — Cliff's delta per cluster (red = higher in mdx)", fontweight="bold")
fig.colorbar(im, label="Cliff's delta")
fig.tight_layout()
fig.savefig(OUT / "fig6_cliff_mdx_wt.png", dpi=200)
plt.show()


## 5. Dystrophin (membrane) — the central defect

In mdx, the median membrane dystrophin drops ~40% in every cluster, and 79-99% of mdx fibers fall below the global WT p25 threshold (vs 14-35% in WT). The defect is *fiber-state-specific in degree* but *universal in direction*.


In [8]:

# Fig 7 — Dystrophin (membrane) per cluster: medians (left) and % of fibers below the
# global WT p25 threshold (right)
dv, do = mval("Dystrophin")
thr = np.percentile(dv[do & (groups == "WT")], 25)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
for k, g in enumerate(GROUPS):
    med = [np.median(dv[(clusters == c) & (groups == g) & do]) for c in range(6)]
    ax.bar(np.arange(6) + (k - 1.5) * 0.2, med, width=0.2, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
ax.set_xticks(range(6), CLUSTERS, fontsize=11)
ax.set_ylabel("median dystrophin (0-255, membrane)")
ax.set_title(f"Dystrophin per cluster", fontweight="bold")
ax.legend(fontsize=9)

ax = axes[1]
for k, g in enumerate(GROUPS):
    frac = [100 * np.mean(dv[(clusters == c) & (groups == g) & do] < thr) for c in range(6)]
    ax.bar(np.arange(6) + (k - 1.5) * 0.2, frac, width=0.2, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
ax.set_xticks(range(6), CLUSTERS, fontsize=11)
ax.set_ylabel("% of fibers below WT p25")
ax.set_title(f"Dystrophin-deficient fibers (threshold = WT p25 = {thr:.0f})", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "fig7_dystrophin_per_cluster.png", dpi=200)
plt.show()


In [9]:

# Fig 8 — per-cluster UMAP panels colored by membrane dystrophin (all groups)
dv, do = mval("Dystrophin")
vmin, vmax = np.nanpercentile(dv[do], [0.5, 99.5])
fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True, sharey=True)
sm = None
for ax, c in zip(axes.ravel(), range(6)):
    m = clusters == c
    umap_scatter(ax, m & ~do, color="white")
    sc = ax.scatter(X_umap[m & do, 0], X_umap[m & do, 1], c=dv[m & do],
                    cmap="viridis", vmin=vmin, vmax=vmax, s=1.0, alpha=0.4,
                    edgecolors="none", rasterized=True)
    ax.text(0.05, 0.95, f"{CLUSTERS[c]} (n={int(m.sum()):,})", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
sm = fig.colorbar(sc, ax=axes.ravel().tolist(), label="dystrophin (membrane)", shrink=0.8)
fig.suptitle("Membrane dystrophin per cluster", fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 0.97, 0.96])
fig.savefig(OUT / "fig8_umap_clusters_dystrophin.png", dpi=200)
plt.show()


/tmp/ipykernel_1440284/3029081267.py:16: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 0.97, 0.96])


## 6. Which correlations are lost in mdx?

Global (all fibers): mdx breaks the healthy couplings **NADH~area** (-0.80 -> -0.56), **DAPI~area** (-0.62 -> -0.39), **Myh7~NADH** (0.88 -> 0.67), **Myh7~COX** (0.82 -> 0.71), **Laminin~WGA** (0.27 -> 0.11, BM-membrane coupling) and **LAMP2~SQSTM1** (0.80 -> 0.70, lysosome-autophagy coupling). It also creates a new negative **Dystrophin~area** coupling (-0.07 -> -0.22): in mdx, larger fibers lose more dystrophin.

Per cluster the losses are state-specific: C2 loses NADH~area (-0.64 -> -0.13) and DAPI~area (-0.48 -> -0.11); C4 loses NADH~area (sign flip) and Myh7~COX; C6 loses Myh7~NADH (0.61 -> 0.24); C5 loses Myh7~NADH (0.53 -> 0.27).


In [10]:

# Fig 9 — correlations lost in mdx
# left: global Spearman rho, WT vs mdx, per pair
# right: per-cluster delta (WT - mdx) for selected pairs
PAIRS = [("NADH", "area"), ("DAPI", "area"), ("Myh7", "NADH"), ("Myh7", "COX"),
         ("Laminin", "WGA"), ("LAMP2", "SQSTM1"), ("IgG", "CD11b"), ("Dystrophin", "area")]

def rho_pair(a, b, m):
    va, oa = mval(a); vb, ob = mval(b)
    o = m & oa & ob
    if o.sum() < 30:
        return np.nan
    r, _ = spearmanr(va[o], vb[o])
    return r

names = [f"{a}~{b}" for a, b in PAIRS]
r_wt = np.array([rho_pair(a, b, groups == "WT") for a, b in PAIRS])
r_mdx = np.array([rho_pair(a, b, groups == "mdx") for a, b in PAIRS])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
ax = axes[0]
y = np.arange(len(PAIRS))
ax.scatter(r_wt, y, s=70, color=GROUP_COLORS["WT"], zorder=3, label="WT")
ax.scatter(r_mdx, y, s=70, color=GROUP_COLORS["mdx"], zorder=3, label="mdx")
for i in range(len(PAIRS)):
    ax.plot([r_wt[i], r_mdx[i]], [i, i], color="0.6", lw=1, zorder=2)
ax.axvline(0, color="0.8", lw=0.8)
ax.set_yticks(y, names, fontsize=9)
ax.set_xlabel("Spearman rho")
ax.set_title("Global: WT vs mdx", fontweight="bold")
ax.legend(fontsize=9)

sel = ["NADH~area", "DAPI~area", "Myh7~NADH", "Myh7~COX", "Laminin~WGA", "Dystrophin~area"]
mat = np.full((6, len(sel)), np.nan)
for j, nm in enumerate(sel):
    a, b = nm.split("~")
    for i in range(6):
        m = clusters == i
        mat[i, j] = rho_pair(a, b, m & (groups == "WT")) - rho_pair(a, b, m & (groups == "mdx"))
ax = axes[1]
im = ax.imshow(mat, aspect="auto", cmap="RdBu", vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(sel)), sel, rotation=45, fontsize=8)
ax.set_yticks(range(6), CLUSTERS, fontsize=11)
for i in range(6):
    for j in range(len(sel)):
        if np.isfinite(mat[i, j]):
            ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Per cluster: rho(WT) - rho(mdx)", fontweight="bold")
fig.colorbar(im, ax=ax, label="correlation lost in mdx")
fig.tight_layout()
fig.savefig(OUT / "fig9_corr_lost_mdx.png", dpi=200)
plt.show()


In [11]:

# Fig 10 — per-cluster UMAP panels colored by IgG (inflammation)
iv, io = mval("IgG")
vmin, vmax = np.nanpercentile(iv[io], [0.5, 99.5])
fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True, sharey=True)
sc = None
for ax, c in zip(axes.ravel(), range(6)):
    m = clusters == c
    umap_scatter(ax, m & ~io, color="white")
    sc = ax.scatter(X_umap[m & io, 0], X_umap[m & io, 1], c=iv[m & io],
                    cmap="magma", vmin=vmin, vmax=vmax, s=1.0, alpha=0.4,
                    edgecolors="none", rasterized=True)
    ax.text(0.05, 0.95, f"{CLUSTERS[c]} (n={int(m.sum()):,})", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
fig.colorbar(sc, ax=axes.ravel().tolist(), label="IgG (whole)", shrink=0.8)
fig.suptitle("IgG per cluster", fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 0.97, 0.96])
fig.savefig(OUT / "fig10_umap_clusters_igg.png", dpi=200)
plt.show()


/tmp/ipykernel_1440284/2757679221.py:16: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 0.97, 0.96])


In [12]:

# Fig 11 — morphology (area) and fiber type (Myh7/Myh2/Myh4) per group
av, ao = mval("area")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
for g in GROUPS:
    v = av[ao & (groups == g)]
    ax.hist(v, bins=60, range=(0, 20000), density=True, alpha=0.55,
            color=GROUP_COLORS[g], label=g)
ax.set_xscale("log")
ax.set_xlabel("fiber area (px^2)")
ax.set_ylabel("density")
ax.set_title("Fiber size distribution", fontweight="bold")
ax.legend(fontsize=9)

ax = axes[1]
mt = ["Myh7", "Myh2", "Myh4"]
w = 0.25
for k, g in enumerate(GROUPS):
    med = [np.median(mval(m)[0][mval(m)[1] & (groups == g)]) for m in mt]
    ax.bar(np.arange(3) + (k - 1.5) * w, med, width=w, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
ax.set_xticks(range(3), mt, fontsize=11)
ax.set_ylabel("median intensity (0-255)")
ax.set_title("Fiber type markers", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "fig11_morph_fibertype.png", dpi=200)
plt.show()


## Answer to Q1 — what is different in mdx?

1. **Dystrophin loss is universal and severe**: median membrane dystrophin 50 -> 31 (Cliff 0.75); 79-99% of mdx fibers are dystrophin-deficient in every cluster. In mdx, residual dystrophin concentrates on intact basement membrane (Laminin~Dystrophin 0.83 -> 0.91) and correlates with stress markers (LAMP2 0.35 -> 0.52, NADH 0.19 -> 0.40) instead of just the BM.
2. **Fiber-state shift**: mdx loses the healthy small-fiber states C4 (37% -> 4%) and C6 (45% -> 7%) and gains pathological states C2 (fibrotic/inflammatory, 4% -> 15%), C3 (small active + immune, 4% -> 38%) and C5 (large Myh4/BM-defect, 0.5% -> 21%).
3. **Inflammation + lysosomal stress in all clusters**: IgG (Cliff -0.62) and CD11b (-0.49) up, LAMP2/LGALS3/SQSTM1 up (slide 8; AAV9 excluded there).
4. **Fiber-type shift**: Myh7 (fast) down, Myh4 (IIx) up — the classic dystrophic fast-to-oxidative transition; NADH rises (compensatory oxidative metabolism) while COX is flat/slightly down.
5. **Basement membrane + membrane integrity**: Laminin and WGA slightly down; the Laminin~WGA coupling is lost (0.27 -> 0.11).
6. **Morphology**: mild hypertrophy (area 5,666 -> 6,126 px^2); size-metabolism and size-nucleus couplings (NADH~area, DAPI~area) are broken — the tissue loses its coordinated scaling.
